In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader

from src.utils.datasets import CUB200Dataset, extract_embeddings

device = "cuda" if torch.cuda.is_available() else "cpu"
device

### CUB200-2011 dataset


In [ ]:
# Download the pretrained ResNet50 model and remove the classifier (last fully
# connected layer) to generate image embeddings for the CUB200 dataset
resnet50 = models.resnet50(pretrained=True)
resnet50.fc = nn.Identity()

In [ ]:
# These transformations preprocess the input images to match the format and statistics
# expected by models pre-trained on ImageNet, such as the ResNet-50
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# The dataset is downloaded if neccessary
train_ds = CUB200Dataset(train=True, transform=transform)
test_ds = CUB200Dataset(train=False, transform=transform)
train_dl = DataLoader(train_ds, batch_size=32, num_workers=4)
test_dl = DataLoader(test_ds, batch_size=32, num_workers=4)

In [ ]:
# Extract output of the ResNet-50's last hidden layer (embeddings) and save them
extract_embeddings(train_dl, resnet50, "cub200_train_embed.pt", device)
extract_embeddings(test_dl, resnet50, "cub200_test_embed.pt", device)